# Protobuf Basics — 09: Protobuf to Parquet Pipeline

Complete production pipeline: Protobuf binary data → normalized Parquet.
This is the standard pattern for consuming gRPC/Kafka Protobuf data.

Topics: batch Protobuf deserialization, schema mapping, normalization,
Parquet output, validation, incremental processing.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Create Simulated Protobuf Landing Zone



In [ ]:

import pathlib, struct as st, random, datetime

LANDING = f"{DATA_DIR}/proto_landing"
pathlib.Path(LANDING).mkdir(exist_ok=True)

def enc_v(v):
    r=b""
    while v>0x7F: r+=bytes([(v&0x7F)|0x80]); v>>=7
    return r+bytes([v&0x7F])
def enc_f(fn,wt,val): return enc_v((fn<<3)|wt)+val
def enc_s(fn,s): e=s.encode(); return enc_f(fn,2,enc_v(len(e))+e)
def enc_i(fn,n): return enc_f(fn,0,enc_v(n))
def enc_d(fn,d): return enc_f(fn,1,st.pack('<d',d))
def make_proto(oid,cid,prod,cat,ctr,qty,price,rev,stat):
    return (enc_i(1,oid)+enc_i(2,cid)+enc_s(3,prod)+enc_s(4,cat)+enc_s(5,ctr)
            +enc_i(6,qty)+enc_d(7,price)+enc_d(8,rev)+enc_s(9,stat))

random.seed(42)
CATS=["Electronics","Books","Clothing","Food","Sports"]
CTRS=["US","UK","DE","FR","JP","CA"]

for day in range(5):
    date=datetime.date(2024,4,1)+datetime.timedelta(days=day)
    n=random.randint(8000,12000)
    rows=[]
    for i in range(n):
        qty=random.randint(1,10); price=round(random.uniform(5,500),2)
        rows.append((bytearray(make_proto(day*20000+i+1,random.randint(1,5000),
            f"Product_{random.randint(1,100)}",random.choice(CATS),random.choice(CTRS),
            qty,price,round(qty*price,2),random.choice(["pending","confirmed","shipped"]))),))
    df_day=spark.createDataFrame(rows,["proto_bytes"])
    df_day.write.mode("overwrite").parquet(f"{LANDING}/date={date}")
    print(f"  {date}: {n:,} binary Protobuf records")


## Step 2 — Deserialize with UDFs



In [ ]:

import struct as st

# UDF-based deserializer for our Order proto format
@F.udf(returnType=StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
]))
def decode_order(b):
    if not b: return None
    d={"order_id":0,"customer_id":0,"product":"","category":"",
       "country":"","quantity":0,"price":0.0,"revenue":0.0,"status":""}
    b=bytes(b); i=0
    while i<len(b):
        if i>=len(b): break
        tag=b[i]; i+=1
        fn=(tag>>3); wt=tag&0x07
        if wt==0:
            v=0; s=0
            while i<len(b):
                byt=b[i]; i+=1; v|=(byt&0x7F)<<s; s+=7
                if not (byt&0x80): break
            if fn==1: d["order_id"]=v
            elif fn==2: d["customer_id"]=v
            elif fn==6: d["quantity"]=v
        elif wt==2:
            ln=b[i]; i+=1; val=b[i:i+ln].decode("utf-8","replace"); i+=ln
            if fn==3: d["product"]=val
            elif fn==4: d["category"]=val
            elif fn==5: d["country"]=val
            elif fn==9: d["status"]=val
        elif wt==1:
            val=st.unpack_from('<d',b,i)[0]; i+=8
            if fn==7: d["price"]=val
            elif fn==8: d["revenue"]=val
        else: break
    return tuple(d.values())

df_all=spark.read.parquet(LANDING)
df_orders=df_all.withColumn("order",decode_order("proto_bytes")).select("order.*")
print(f"Deserialized: {df_orders.count():,} orders")
df_orders.show(5)


## Step 3 — Write to Parquet Storage



In [ ]:

STORAGE = f"{DATA_DIR}/proto_storage"
df_orders.write.mode("overwrite") \
              .option("compression","zstd") \
              .partitionBy("category") \
              .parquet(STORAGE)

pq_count=spark.read.parquet(STORAGE).count()
print(f"Parquet storage: {pq_count:,} rows (partitioned by category)")

# Verify
proto_count=spark.read.parquet(LANDING).count()
print(f"Source count : {proto_count:,}")
print(f"Target count : {pq_count:,}")
print(f"Match        : {'✅' if proto_count==pq_count else '❌'}")

spark.read.parquet(STORAGE).groupBy("category").agg(
    F.count("*").alias("orders"),
    F.sum("revenue").alias("revenue")
).orderBy(F.desc("revenue")).show()


## Step 4 — Production Pipeline Template



In [ ]:

print("""
Production Protobuf → Parquet pipeline template:

  from pyspark.sql.protobuf.functions import from_protobuf

  def proto_to_parquet(
      input_path:  str,   # binary Protobuf files or Kafka
      output_path: str,   # Parquet output
      desc_file:   str,   # path to .desc compiled descriptor
      msg_name:    str,   # Protobuf message name
      partition_col: str, # column to partition output by
  ):
      # 1. Read binary data
      df_raw = spark.read.parquet(input_path)

      # 2. Deserialize Protobuf → struct
      df_orders = df_raw.select(
          from_protobuf(col("proto_bytes"), msg_name, desc_file).alias("o")
      ).select("o.*")

      # 3. Add derived columns
      df_enriched = df_orders \
          .withColumn("_ingested_at", current_timestamp()) \
          .withColumn("revenue", col("price") * col("quantity"))

      # 4. Validate
      count = df_enriched.count()

      # 5. Write partitioned Parquet
      df_enriched.write.mode("overwrite") \
                 .option("compression", "zstd") \
                 .partitionBy(partition_col) \
                 .parquet(output_path)

      return count
""")


## Summary



In [ ]:

print("""
Protobuf → Parquet pipeline summary:
  1. Source: Kafka topic or binary files containing serialized Protobuf messages
  2. Deserialize: from_protobuf(col("bytes"), "MessageName", "path.desc")
  3. Normalize: add derived columns, handle oneof variants, flatten nested
  4. Write: partitioned Parquet with zstd compression
  5. Validate: row count match between source and target

Protobuf-specific considerations:
  - Compile .proto to .desc file (protoc or grpcio-tools)
  - Default values (0, "", false) appear for missing optional fields
  - Enum fields are integer values — map to strings if needed
  - Timestamp fields are StructType(seconds, nanos) — cast to TimestampType
  - oneof: only one variant is non-null at a time
""")
